In [1]:
import math
import torch
from torch import nn
from torch.nn import init
from torch.nn import functional as F
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import math
import torch.nn.functional as F
from torch.utils.data import Subset
import numpy as np
import pandas as pd
from tqdm import tqdm
import copy
from torch.utils.data import ConcatDataset

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [3]:
class TimeEmbed(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(1, dim),
            nn.SiLU(),
            nn.Linear(dim, dim)
        )

    def forward(self, t):
        return self.mlp(t[:, None])


In [4]:
class Block(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_emb = nn.Linear(t_dim, out_ch)

    def forward(self, x, t):
        h = F.silu(self.conv1(x))
        h = h + self.time_emb(t)[:, :, None, None]
        h = F.silu(self.conv2(h))
        return h

class StandardClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        base = 32
        t_dim = 32

        self.time_embed = TimeEmbed(t_dim)

        self.down1 = Block(3, base, t_dim)
        self.down2 = Block(base, base * 2, t_dim)
        self.down3 = Block(base * 2, base * 2, t_dim)
        self.pool = nn.MaxPool2d(2)

        self.mid = Block(base * 2, base * 2, t_dim)

        self.out = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(base * 2, 2)
        )
        
    def forward(self, x, t):
        t = self.time_embed(t)

        d1 = self.down1(x, t)
        d2 = self.down2(self.pool(d1), t)
        d3 = self.down3(self.pool(d2), t)

        m = self.mid(d3, t)

        return self.out(m)

standardClassifier = StandardClassifier()

print(sum(p.numel() for p in standardClassifier.parameters()))

standardClassifier(torch.randn(5, 3, 32, 32), torch.randn((5,)))

221922


tensor([[-0.0784, -0.0372],
        [-0.0796, -0.0309],
        [-0.0784, -0.0378],
        [-0.0783, -0.0489],
        [-0.0782, -0.0466]], grad_fn=<AddmmBackward0>)

In [5]:
T = 1000
beta = torch.linspace(1e-4, 0.02, T).to(device)
alpha = 1.0 - beta
alpha_bar = torch.cumprod(alpha, dim=0)

In [6]:
from datasets import load_dataset
from torch.utils.data import DataLoader
import torch
from torchvision import transforms
dataset = load_dataset("cifar10")


In [7]:
animal_classes = {2, 3, 4, 5, 6, 7}  

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

def preprocess(batch):
    batch["pixel_values"] = [transform(img) for img in batch["img"]]
    batch["label"] = [
        1 if label in animal_classes else 0
        for label in batch["label"]
    ]
    return batch


dataset = dataset.with_transform(preprocess)

def collate_fn(batch):
    images = torch.stack([item["pixel_values"] for item in batch])
    labels = torch.tensor([item["label"] for item in batch])
    return images, labels

train_loader = DataLoader(
    dataset["train"],
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    dataset["test"],
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)



In [ ]:
# for x, y in tqdm(train_loader, desc=f"Epoch {0} [Train]"):
#     print(x.mean())
#     break

In [8]:
standardClassifier = standardClassifier.to(device)
opt_clf = torch.optim.Adam(standardClassifier.parameters(), lr=1e-3)
ce = nn.CrossEntropyLoss()

EPOCHS = 50

for epoch in range(EPOCHS):
    # ---------- TRAIN ----------
    standardClassifier.train()
    train_correct = train_total = 0

    for x, y in tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
        x, y = x.to(device), y.to(device)
        B = x.size(0)

        t = torch.randint(0, T, (B,), device=device)
        noise = torch.randn_like(x)
        a = alpha_bar[t].view(-1, 1, 1, 1)
        x_noisy = torch.sqrt(a) * x + torch.sqrt(1 - a) * noise

        logits = standardClassifier(x_noisy, t / T)
        loss = ce(logits, y)

        opt_clf.zero_grad()
        loss.backward()
        opt_clf.step()

        train_correct += (logits.argmax(1) == y).sum().item()
        train_total += y.size(0)

    train_acc = 100 * train_correct / train_total

    # ---------- TEST ----------
    standardClassifier.eval()
    test_correct = test_total = 0
    test_loss = 0.0

    with torch.no_grad():
        for x, y in tqdm(test_loader, desc=f"Epoch {epoch} [Test]"):
            x, y = x.to(device), y.to(device)
            B = x.size(0)

            t = torch.randint(0, T, (B,), device=device)
            noise = torch.randn_like(x)
            a = alpha_bar[t].view(-1, 1, 1, 1)
            x_noisy = torch.sqrt(a) * x + torch.sqrt(1 - a) * noise

            logits = standardClassifier(x_noisy, t / T)
            loss = ce(logits, y)

            test_loss += loss.item() * B
            test_correct += (logits.argmax(1) == y).sum().item()
            test_total += B

    test_acc = 100 * test_correct / test_total
    test_loss /= test_total

    print(
        f"Epoch {epoch:03d} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Test Acc: {test_acc:.2f}% | "
        f"Test Loss: {test_loss:.4f}"
    )


Epoch 0 [Test]: 100%|██████████| 157/157 [00:03<00:00, 51.84it/s]


Epoch 000 | Train Acc: 69.30% | Test Acc: 74.34% | Test Loss: 0.5009


Epoch 1 [Test]: 100%|██████████| 157/157 [00:03<00:00, 48.99it/s]


Epoch 001 | Train Acc: 74.23% | Test Acc: 74.43% | Test Loss: 0.4902


Epoch 2 [Test]: 100%|██████████| 157/157 [00:02<00:00, 53.67it/s]


Epoch 002 | Train Acc: 74.56% | Test Acc: 75.82% | Test Loss: 0.4782


... (77 outputs left out)

Epoch 041 | Train Acc: 77.30% | Test Acc: 77.48% | Test Loss: 0.4506


Epoch 42 [Test]: 100%|██████████| 157/157 [00:03<00:00, 49.34it/s]


Epoch 042 | Train Acc: 77.66% | Test Acc: 77.48% | Test Loss: 0.4446


Epoch 43 [Train]:  86%|████████▌ | 669/782 [00:16<00:02, 41.33it/s]

KeyboardInterrupt



In [9]:
torch.save(standardClassifier.state_dict(), "standardClassifier.pth")